# Experiment tracking with MLflow

Dilnoza and Sam tune a crop-yield predictor for a farm co-op, and nobody can say which of last week's six runs
was the good one. You will replay that messy week, write a ten-line tracker that keeps a record of every run,
then log the same six runs with MLflow and compare them with `search_runs`. Same invented data and same code as
the lesson.

About 4 minutes. No account, key or server needed. Run the cells from top to bottom.

## 1. Install MLflow

This installs the small client version of MLflow (`mlflow-skinny`), plus the two database libraries MLflow uses
for its default local store, a SQLite file. These are the versions the lesson was run with.

**Look for:** no error. The install takes about 20 seconds.

In [1]:
%pip install -q mlflow-skinny==3.18.0 sqlalchemy==2.1.8 alembic==1.20.1

Note: you may need to restart the kernel to use updated packages.


## 2. The farm data and the training loop

This is the team's training code. The data is invented: 120 field-seasons drawn with seed 0, each with rain
(mm), fertiliser (kg/ha), mean temperature (C), a soil score (1 to 10) and the yield in tonnes per hectare.
90 rows are for training and 30 for validation. `train` fits a linear model by gradient descent and returns the
error on both row sets (RMSE, in tonnes per hectare; lower is better). `WEEK` holds the six settings the team
tried last week, in order.

**Look for:** 90 training rows and 30 validation rows, and 6 settings in `WEEK`.

In [2]:
import numpy as np

FEATURES = ["rain", "fertiliser", "temp", "soil"]


def make_data(seed=0, n=120):
    rng = np.random.default_rng(seed)
    rain = rng.uniform(300, 900, n)
    fert = rng.uniform(20, 160, n)
    temp = rng.uniform(18, 30, n)
    soil = rng.integers(1, 11, n).astype(float)
    yield_t = 0.5 + 0.004 * rain + 0.012 * fert - 0.06 * temp + 0.15 * soil + rng.normal(0, 0.35, n)
    X = np.column_stack([rain, fert, temp, soil])
    order = rng.permutation(n)
    train, val = order[:90], order[90:]
    return X[train], yield_t[train], X[val], yield_t[val]


def rmse(pred, y):
    return float(np.sqrt(np.mean((pred - y) ** 2)))


def train(lr, epochs, features, on_epoch=None):
    """Linear model trained by gradient descent on mean squared error.
    Returns (weights, bias, mean, std, train_rmse, val_rmse). on_epoch(epoch, train_rmse) is called every epoch."""
    Xtr, ytr, Xva, yva = make_data()
    cols = [FEATURES.index(f) for f in features]
    Xtr, Xva = Xtr[:, cols], Xva[:, cols]
    mean, std = Xtr.mean(axis=0), Xtr.std(axis=0)
    Ztr, Zva = (Xtr - mean) / std, (Xva - mean) / std
    w, b = np.zeros(len(cols)), 0.0
    with np.errstate(over="ignore", invalid="ignore"):
        for epoch in range(1, epochs + 1):
            err = Ztr @ w + b - ytr
            w = w - lr * 2 * Ztr.T @ err / len(ytr)
            b = b - lr * 2 * err.mean()
            if on_epoch:
                on_epoch(epoch, rmse(Ztr @ w + b, ytr))
        return w, b, mean, std, rmse(Ztr @ w + b, ytr), rmse(Zva @ w + b, yva)


# The six settings the team tried last week, in the order they tried them.
WEEK = [
    {"lr": 0.01, "epochs": 50, "features": ["rain", "fertiliser"]},
    {"lr": 0.01, "epochs": 300, "features": ["rain", "fertiliser"]},
    {"lr": 0.1, "epochs": 300, "features": ["rain", "fertiliser"]},
    {"lr": 0.1, "epochs": 300, "features": ["rain", "fertiliser", "soil"]},
    {"lr": 0.1, "epochs": 300, "features": ["rain", "fertiliser", "temp", "soil"]},
    {"lr": 1.2, "epochs": 300, "features": ["rain", "fertiliser", "temp", "soil"]},
]

Xtr, ytr, Xva, yva = make_data()
print("training rows:", len(ytr), "| validation rows:", len(yva))
print("settings in WEEK:", len(WEEK))

training rows: 90 | validation rows: 30
settings in WEEK: 6


## 3. Last week, the untracked way

The six runs, each saved under the file name the person typed that day (Monday, Monday, Tuesday, Wednesday,
Thursday, Friday). The files go into a folder `messy/`.

**Look for:** runs 4, 5 and 6 are all saved as `yield_model_final.npz`, so the folder holds only 4 files. Run 5
is the best (val_rmse 0.3552), but its file was overwritten by run 6, whose error is about 6.3e+93: training
blew up. The weights left in `final` are huge numbers.

In [3]:
import os
import shutil

OUT = "messy"
shutil.rmtree(OUT, ignore_errors=True)
os.makedirs(OUT)

NAMES = ["yield_model.npz", "yield_model_v2.npz", "yield_model_lr01.npz",
         "yield_model_final.npz", "yield_model_final.npz", "yield_model_final.npz"]

for n, (settings, name) in enumerate(zip(WEEK, NAMES), start=1):
    w, b, mean, std, train_rmse, val_rmse = train(**settings)
    path = os.path.join(OUT, name)
    existed = os.path.exists(path)
    np.savez(path, w=w, b=b, mean=mean, std=std)
    print(f"run {n}: {settings} -> train_rmse={train_rmse:.4g} val_rmse={val_rmse:.4g}; "
          f"saved {name}{' (overwrote the old file)' if existed else ''}")

print("folder:", sorted(os.listdir(OUT)))
final = np.load(os.path.join(OUT, "yield_model_final.npz"))
print("yield_model_final.npz weights:", final["w"].tolist())

run 1: {'lr': 0.01, 'epochs': 50, 'features': ['rain', 'fertiliser']} -> train_rmse=1.506 val_rmse=1.367; saved yield_model.npz
run 2: {'lr': 0.01, 'epochs': 300, 'features': ['rain', 'fertiliser']} -> train_rmse=0.6412 val_rmse=0.5725; saved yield_model_v2.npz
run 3: {'lr': 0.1, 'epochs': 300, 'features': ['rain', 'fertiliser']} -> train_rmse=0.6411 val_rmse=0.5743; saved yield_model_lr01.npz
run 4: {'lr': 0.1, 'epochs': 300, 'features': ['rain', 'fertiliser', 'soil']} -> train_rmse=0.4141 val_rmse=0.3984; saved yield_model_final.npz
run 5: {'lr': 0.1, 'epochs': 300, 'features': ['rain', 'fertiliser', 'temp', 'soil']} -> train_rmse=0.3735 val_rmse=0.3552; saved yield_model_final.npz (overwrote the old file)
run 6: {'lr': 1.2, 'epochs': 300, 'features': ['rain', 'fertiliser', 'temp', 'soil']} -> train_rmse=7.281e+93 val_rmse=6.322e+93; saved yield_model_final.npz (overwrote the old file)
folder: ['yield_model.npz', 'yield_model_final.npz', 'yield_model_lr01.npz', 'yield_model_v2.npz']


## 4. A tracker in ten lines

`log_run` is the function from the lesson. After each run it gives the run an id, copies the model file into the
run's own folder, and appends one JSON line to `runs.jsonl`. The training loop saves `model.npz` under the same
name every time, on purpose: the copy is what keeps each run's model.

**Look for:** six lines in `runs.jsonl`, `run1` to `run6`, each with its settings, two named scores and its own
`model_file`. The best line by val_rmse is `run5` with 0.3552, and six model copies are kept, one per run folder.

In [4]:
import json

shutil.rmtree("runs", ignore_errors=True)
if os.path.exists("runs.jsonl"):
    os.remove("runs.jsonl")


def read_runs():
    if not os.path.exists("runs.jsonl"):
        return []
    with open("runs.jsonl") as f:
        return [json.loads(line) for line in f]


def log_run(params, metrics, model_file):
    run_id = f"run{len(read_runs()) + 1}"
    folder = os.path.join("runs", run_id)
    os.makedirs(folder)
    kept = shutil.copy(model_file, folder)
    record = {"run_id": run_id, "params": params,
              "metrics": metrics, "model_file": kept}
    with open("runs.jsonl", "a") as f:
        f.write(json.dumps(record) + "\n")
    return run_id


for settings in WEEK:
    w, b, mean, std, train_rmse, val_rmse = train(**settings)
    np.savez("model.npz", w=w, b=b, mean=mean, std=std)   # same file name every time, on purpose
    metrics = {"train_rmse": round(train_rmse, 4), "val_rmse": round(val_rmse, 4)}
    log_run(settings, metrics, "model.npz")

print("runs.jsonl:")
with open("runs.jsonl") as f:
    print(f.read(), end="")
best = min(read_runs(), key=lambda r: r["metrics"]["val_rmse"])
print("best:", json.dumps(best))
print("kept files:", sorted(os.path.join(d, f) for d, _, fs in os.walk("runs") for f in fs))

runs.jsonl:
{"run_id": "run1", "params": {"lr": 0.01, "epochs": 50, "features": ["rain", "fertiliser"]}, "metrics": {"train_rmse": 1.5059, "val_rmse": 1.3669}, "model_file": "runs/run1/model.npz"}
{"run_id": "run2", "params": {"lr": 0.01, "epochs": 300, "features": ["rain", "fertiliser"]}, "metrics": {"train_rmse": 0.6412, "val_rmse": 0.5725}, "model_file": "runs/run2/model.npz"}
{"run_id": "run3", "params": {"lr": 0.1, "epochs": 300, "features": ["rain", "fertiliser"]}, "metrics": {"train_rmse": 0.6411, "val_rmse": 0.5743}, "model_file": "runs/run3/model.npz"}
{"run_id": "run4", "params": {"lr": 0.1, "epochs": 300, "features": ["rain", "fertiliser", "soil"]}, "metrics": {"train_rmse": 0.4141, "val_rmse": 0.3984}, "model_file": "runs/run4/model.npz"}
{"run_id": "run5", "params": {"lr": 0.1, "epochs": 300, "features": ["rain", "fertiliser", "temp", "soil"]}, "metrics": {"train_rmse": 0.3735, "val_rmse": 0.3552}, "model_file": "runs/run5/model.npz"}
{"run_id": "run6", "params": {"lr": 1.

## 5. Start MLflow

With no setup, MLflow keeps its run records in a SQLite file, `mlflow.db`, in the folder the notebook runs from,
and the files attached to runs in a folder `mlruns/` next to it. No server runs. The first use creates the
database, so MLflow prints a few INFO lines; that is normal.

If you run this notebook a second time in the same session, this cell marks the old `crop-yield` runs as deleted
first, so the searches below see only one set of six runs.

**Look for:** `tracking uri: sqlite:///<working folder>/mlflow.db` and `active runs in crop-yield: 0`.

In [5]:
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"   # keeps MLflow from printing a long hint for coding assistants
import pandas as pd
import mlflow
from mlflow import MlflowClient

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)
client = MlflowClient()

print("tracking uri:", mlflow.get_tracking_uri().replace(os.getcwd(), "<working folder>"))
old = mlflow.get_experiment_by_name("crop-yield")
if old:
    for r in client.search_runs([old.experiment_id]):
        client.delete_run(r.info.run_id)
print("active runs in crop-yield:", 0 if old is None else len(client.search_runs([old.experiment_id])))

2026/11/14 15:28:36 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/11/14 15:28:36 INFO mlflow.store.db.utils: Updating database tables


tracking uri: sqlite:///<working folder>/mlflow.db
active runs in crop-yield: 0


## 6. Log the six runs with MLflow

This is the loop from the lesson, line for line. `log_params` stores the settings, `log_metric` with `step=epoch`
adds one train_rmse point per epoch, `log_metric("val_rmse", ...)` stores the final score, and `log_artifact`
copies `model.npz` into the run's own folder. It writes 1,556 metric points, so it takes about 10 seconds.

**Look for:** one line per run with its id. MLflow gives each run a long random id, so yours differ from the
lesson's.

In [6]:
mlflow.set_experiment("crop-yield")
def log_curve(epoch, loss):
    mlflow.log_metric("train_rmse", loss,
                      step=epoch)
for n, settings in enumerate(WEEK, start=1):
    mlflow.start_run(run_name=f"run{n}")
    cols = ",".join(settings["features"])
    mlflow.log_params({**settings,
                       "features": cols})
    w, b, mean, std, _, val_rmse = train(
        **settings, on_epoch=log_curve)
    mlflow.log_metric("val_rmse", val_rmse)
    np.savez("model.npz", w=w, b=b,
             mean=mean, std=std)
    mlflow.log_artifact("model.npz")
    mlflow.end_run()
    print(f"logged run{n}:", mlflow.last_active_run().info.run_id)

2026/11/14 15:28:38 INFO mlflow.tracking.fluent: Experiment with name 'crop-yield' does not exist. Creating a new experiment.


logged run1: 4a31ee498d954b2b8b4ed694e3b4fbb8


logged run2: dc64a50e719f450395da34ad21663b09


logged run3: aa175d9184f341649857db6db2217c97


logged run4: d6db17cdde1e4d328097d4eb5c0dfa48


logged run5: 59fe271e49334ebd95689d05dc5aecfa


logged run6: 8b0cdfe6412d4a549ad0f8a5f8186cd6


## 7. All runs, best first

`search_runs` returns the runs as a pandas table: one row per run, a `params.` column per setting and a
`metrics.` column per score. `order_by` sorts by val_rmse, smallest error first. pandas prints the scores in
scientific notation: `3.551657e-01` means 0.3552.

**Look for:** 6 rows. `run5` is first with val_rmse 3.551657e-01. `run6` is last with 6.322268e+93: the run that
blew up was still logged.

In [7]:
cols = ["tags.mlflow.runName", "params.lr", "params.epochs", "params.features", "metrics.train_rmse", "metrics.val_rmse"]
runs = mlflow.search_runs(experiment_names=["crop-yield"], order_by=["metrics.val_rmse ASC"])
print("rows:", len(runs))
print("columns:", sorted(c for c in runs.columns if c.startswith(("params.", "metrics."))))
print(runs[cols].to_string(index=False))

rows: 6
columns: ['metrics.train_rmse', 'metrics.val_rmse', 'params.epochs', 'params.features', 'params.lr']
tags.mlflow.runName params.lr params.epochs           params.features  metrics.train_rmse  metrics.val_rmse
               run5       0.1           300 rain,fertiliser,temp,soil        3.734771e-01      3.551657e-01
               run4       0.1           300      rain,fertiliser,soil        4.140803e-01      3.984053e-01
               run2      0.01           300           rain,fertiliser        6.412425e-01      5.724937e-01
               run3       0.1           300           rain,fertiliser        6.411361e-01      5.743392e-01
               run1      0.01            50           rain,fertiliser        1.505856e+00      1.366850e+00
               run6       1.2           300 rain,fertiliser,temp,soil        7.281440e+93      6.322268e+93


## 8. Filter: lr 0.1 and val_rmse under 0.5

Sam wants only the runs with lr 0.1 whose val_rmse is under 0.5. MLflow stores every setting as text, so the
setting is compared with a quoted string, `"0.1"`. Scores are numbers, so `< 0.5` has no quotes.

**Look for:** 2 rows, `run5` (0.355166) and `run4` (0.398405).

In [8]:
q = 'params.lr = "0.1" AND metrics.val_rmse < 0.5'
print("filter_string:", q)
print(mlflow.search_runs(experiment_names=["crop-yield"], filter_string=q,
                         order_by=["metrics.val_rmse ASC"])[cols].to_string(index=False))

filter_string: params.lr = "0.1" AND metrics.val_rmse < 0.5
tags.mlflow.runName params.lr params.epochs           params.features  metrics.train_rmse  metrics.val_rmse
               run5       0.1           300 rain,fertiliser,temp,soil            0.373477          0.355166
               run4       0.1           300      rain,fertiliser,soil            0.414080          0.398405


## 9. Two filters that fail, on purpose

The first writes the setting as a number, the second joins conditions with `OR`. Both are the wrong answers from
the lesson's screen 7. Each search is wrapped in `try/except`, so the error is printed and the notebook goes on.

**Look for:** `Expected a quoted string value for parameter` for the first, and
`Invalid clause(s) in filter string: 'OR'` for the second. MLflow's search joins conditions with `AND` only.

In [9]:
for q in ["params.lr = 0.1",
          'params.lr = "0.1" OR metrics.val_rmse < 0.5']:
    print("filter_string:", q)
    try:
        print(mlflow.search_runs(experiment_names=["crop-yield"], filter_string=q)[cols].to_string(index=False))
    except Exception as e:
        print(type(e).__name__ + ":", e)
    print()

filter_string: params.lr = 0.1
MlflowException: Expected a quoted string value for parameter (e.g. 'my-value'). Got value 0.1

filter_string: params.lr = "0.1" OR metrics.val_rmse < 0.5
MlflowException: Invalid clause(s) in filter string: 'OR'



## 10. The best run's model file and code version

The best run is the first row of the sorted table. Its artifacts are the files attached to it; here, its own copy
of `model.npz`. MLflow also stores tags, short text labels it adds by itself. `mlflow.source.git.commit` is the
git commit the code ran from, but MLflow only adds it when the code runs inside a git repository. This notebook is
not in one, so that tag is missing.

**Look for:** `artifacts: ['model.npz']`, an artifact path that ends in `/artifacts` inside `mlruns/`, and
`git commit tag: None`.

In [10]:
best = runs.iloc[0]
print("best run:", best["tags.mlflow.runName"], "| val_rmse:", round(best["metrics.val_rmse"], 4))
print("artifact_uri:", best["artifact_uri"].replace(os.getcwd(), "<working folder>"))
print("artifacts:", [a.path for a in client.list_artifacts(best["run_id"])])
print("git commit tag:", client.get_run(best["run_id"]).data.tags.get("mlflow.source.git.commit"))

best run: run5 | val_rmse: 0.3552
artifact_uri: <working folder>/mlruns/1/59fe271e49334ebd95689d05dc5aecfa/artifacts
artifacts: ['model.npz']
git commit tag: None


## 11. The train_rmse curve of each run

Each `log_metric("train_rmse", loss, step=epoch)` call added one point, so a run has one point per epoch.
`get_metric_history` reads them back. Below are the points at epochs 1, 50, 100, 200 and 300.

**Look for:** run1 has 50 points and run2 has 300. Both start at 3.6703 and reach 1.5059 at epoch 50. run1 stopped
there while its error was still falling; run2 went on to 0.6412.

In [11]:
def curve(name):
    rid = runs[runs["tags.mlflow.runName"] == name]["run_id"].iloc[0]
    hist = sorted(client.get_metric_history(rid, "train_rmse"), key=lambda m: m.step)
    return len(hist), [(m.step, round(m.value, 4)) for m in hist if m.step in (1, 50, 100, 200, 300)]

for name in ["run1", "run2"]:
    points, at = curve(name)
    print(name, "points:", points, at)

run1 points: 50 [(1, 3.6703), (50, 1.5059)]
run2 points: 300 [(1, 3.6703), (50, 1.5059), (100, 0.818), (200, 0.6454), (300, 0.6412)]


## Now change it

Three small experiments. Each cell has one line marked `# <- change this`. Edit it, run the cell, and compare.

### Experiment 1: loosen the filter

Change `0.5` to `0.6`. `run3` (lr 0.1, val_rmse 0.5743) should join the result, so you get 3 rows instead of 2.
`run2` stays out even though its score is under 0.6, because its lr is 0.01: both conditions must hold.

In [12]:
LIMIT = 0.5  # <- change this: 0.6
q = f'params.lr = "0.1" AND metrics.val_rmse < {LIMIT}'
print("filter_string:", q)
found = mlflow.search_runs(experiment_names=["crop-yield"], filter_string=q, order_by=["metrics.val_rmse ASC"])
print("rows:", len(found))
print(found[cols].to_string(index=False))

filter_string:

 params.lr = "0.1" AND metrics.val_rmse < 0.5


rows: 2
tags.mlflow.runName params.lr params.epochs           params.features  metrics.train_rmse  metrics.val_rmse
               run5       0.1           300 rain,fertiliser,temp,soil            0.373477          0.355166
               run4       0.1           300      rain,fertiliser,soil            0.414080          0.398405


### Experiment 2: look at the curve of the run that blew up

Change `"run5"` to `"run6"`. run5's error falls from 3.006 and has settled at 0.3735 by epoch 50. run6 (lr 1.2)
should do the opposite: each step overshoots further, so the error grows from 5.1994 at epoch 1 to about 3.6e+14
at epoch 50 and 7.3e+93 at epoch 300. The
curve shows you *when* it went wrong, which a single final score cannot.

In [13]:
NAME = "run5"  # <- change this: "run6"
points, at = curve(NAME)
print(NAME, "points:", points, at)

run5

 points: 300 [(1, 3.0058), (50, 0.3735), (100, 0.3735), (200, 0.3735), (300, 0.3735)]


### Experiment 3: log a seventh run

This logs one new run with run5's settings, except the number of epochs. As it stands, `run7` repeats run5
exactly and gets the same val_rmse, 0.3552: same data, same seed, same settings, same score. Change `300` to `10`.
With only 10 steps the model has not reached its low point yet, so val_rmse should come out at about 0.5763, and
`run7` should drop from the top of the table to fifth place, just below run3. The new run is kept either way: every
run gets a record. (Run this cell again and you add another `run7`: MLflow never overwrites a run.)

In [14]:
EPOCHS = 300  # <- change this: 10
settings = {"lr": 0.1, "epochs": EPOCHS, "features": ["rain", "fertiliser", "temp", "soil"]}
with mlflow.start_run(run_name="run7"):
    mlflow.log_params({**settings, "features": ",".join(settings["features"])})
    w, b, mean, std, _, val_rmse = train(**settings, on_epoch=log_curve)
    mlflow.log_metric("val_rmse", val_rmse)
print("run7 val_rmse:", round(val_rmse, 4))
table = mlflow.search_runs(experiment_names=["crop-yield"], order_by=["metrics.val_rmse ASC"])
print(table[["tags.mlflow.runName", "params.epochs", "metrics.val_rmse"]].to_string(index=False))

run7 val_rmse: 0.3552
tags.mlflow.runName params.epochs  metrics.val_rmse
               run7           300      3.551657e-01
               run5           300      3.551657e-01
               run4           300      3.984053e-01
               run2           300      5.724937e-01
               run3           300      5.743392e-01
               run1            50      1.366850e+00
               run6           300      6.322268e+93


## What you should see

- Cell 3: runs 4, 5 and 6 all save to `yield_model_final.npz`; the folder holds 4 files and `final` holds run 6's huge weights.
- Cell 4: `runs.jsonl` has 6 lines and the best is `run5` with val_rmse 0.3552, with its own copy `runs/run5/model.npz`.
- Cell 7: the experiment `crop-yield` holds 6 runs, `run1` to `run6`. Sorted by `metrics.val_rmse`, `run5` comes first with 3.551657e-01 (about 0.3552), and `run6` is last with about 6.3e+93: the run with lr 1.2 blew up and was still logged.
- Cell 9: `params.lr = 0.1` fails with "Expected a quoted string value"; cell 8, with `"0.1"` in quotes, returns 2 runs.
- Cell 10: the best run's artifacts list shows `model.npz`.
- Cell 11: the train_rmse history has 50 points for run1 and 300 for run2.